In [ ]:
import functools
import os

from gossiplearning.config import Config
from utils.model_creators import create_MLP
from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from utils.data import load_npz_data
simulation_number = 0
num_nodes = 10
k = 3
aggr = "ImprovedOverwrite"
alpha = 5

#4inTestBalanced
#model = tuple([torch.load(f"experiments/SE-CIC-IDS2018/{simulation_number}/models/{i}.pth") for i in range(num_nodes)])
#datasets = tuple([load_npz_data(f"data/datasets/SE-CIC-IDS2018_{num_nodes}n_{k}k/{simulation_number}/4inTestSubSampledAlpha05/node_{i}.npz") for i in range(num_nodes)])

model = tuple([torch.load(f"experiments/SE-CIC-IDS2018/{num_nodes}NodesAlpha0{alpha}{aggr}/{simulation_number}/models/{i}.pth") for i in range(num_nodes)])
datasets = tuple([load_npz_data(f"data/datasets/SE-CIC-IDS2018_{num_nodes}n_{k}k/{simulation_number}/4inTestBalancedAlpha0{alpha}/node_{i}.npz") for i in range(num_nodes)])


X = np.concatenate([d[4] for d in datasets])
Y = np.concatenate([d[5] for d in datasets])

#n_samples = 500000
#n_samples = 100000
#idx = np.random.choice(len(X), size=n_samples, replace=False)

In [ ]:
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from utils.metrics import compute_metrics

import seaborn as sns

class MLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        input_dim = 9
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.output_layer = nn.Linear(64, 12)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logit = self.output_layer(x)  
        return logit


x_tensor = torch.tensor(X.astype("float32"), dtype=torch.float32)
y_tensor = torch.tensor(Y, dtype=torch.long)
num_classes = 12
#num_nodes = 20
error_matrix = np.zeros((num_nodes, num_classes))

print("X: ", x_tensor.shape)
print("Y: ", y_tensor.shape)
accuracy_local_list = []
history = []

for i in range(num_nodes):
    print("Initializing model from node ", i)
    loaded_model = MLPModel()
    loaded_model.load_state_dict(model[i])

    loaded_model.eval()
    with torch.no_grad():
        logits = loaded_model(x_tensor)              
        y_pred = torch.argmax(logits, dim=1)         
        correct = (y_pred == y_tensor).sum().item()
        accuracy_local = correct / len(y_tensor)

    print("Node: ", str(i))
    print("Accuracy:", str(accuracy_local), "\n")
    accuracy_local_list.append(accuracy_local)
    probs = torch.softmax(logits, dim=1)

    predictions_loc = y_pred.numpy()
    true_pred = y_tensor.numpy()
    metrics = compute_metrics(true_pred, predictions_loc)
    dict_history = {
        "Node": str(i),
        "accuracy": metrics.acc,
        "precision_macro": metrics.prec,
        "recall_macro": metrics.rec,
        "f1_macro": metrics.f1,
        "f1_weighted": metrics.f1_weighted
    }
    history.append(dict_history)
    target_names = [str(c) for c in np.unique(true_pred)]
    report = classification_report(true_pred, 
                                        predictions_loc, 
                                        labels=range(num_classes), 
                                        target_names=target_names, zero_division=0,
                                        output_dict=True
                                        )

    cm = confusion_matrix(true_pred, predictions_loc, labels=range(num_classes))
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                        xticklabels=range(num_classes), yticklabels=range(num_classes))
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    #path_folder = Path(f"experiments/SE-CIC-IDS2018/{simulation_number}/plots/testing_unified")
    path_folder = Path(f"experiments/SE-CIC-IDS2018/{num_nodes}NodesAlpha0{alpha}{aggr}") / f"{simulation_number}" / "testing_unified"
    path_folder.mkdir(parents=True, exist_ok=True)
    plt.savefig(path_folder/ f"matrix_node{i}.png")
    plt.close()

    file_path = path_folder / "report_accuracy.json"
    if not file_path.exists():
        with open(file_path, "w") as outfile:
            json.dump(dict_history, outfile, indent=1)
    else:
        with open(file_path, "r+") as outfile:
            file_data = json.load(outfile)
            if not isinstance(file_data, list):
                file_data = [file_data]
            file_data.append(dict_history)
            outfile.seek(0)
            json.dump(file_data, outfile, indent=1)
        outfile.close
    with open(path_folder/ f"report_node_{i}.json", "w") as outfiler:
                json.dump(report, outfiler, indent=3)
    outfiler.close
    for c in range(num_classes):
        mask = (true_pred == c)
        errors = np.sum(predictions_loc[mask] != c)
        total = np.sum(mask)
        error_matrix[i, c] = errors / total if total > 0 else 0.0

# Plot
x = np.arange(num_nodes)  # nodi
width = 0.1               # larghezza delle barre
plt.figure(figsize=(25,10))

for c in range(num_classes):
    plt.bar(x + c*width, error_matrix[:, c], width, label=f"Classe {c}")

plt.xlabel("Nodo")
plt.ylabel("Errore (%)")
plt.title("Percentuale di errore con α = 0.5")
plt.xticks(x + (num_classes/2 - 0.5)*width, [f"Nodo {i}" for i in range(num_nodes)])
plt.legend()
plt.tight_layout()
plt.savefig(path_folder/ f"istogramma_err.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

xs = [x for x in range(len(history))]

all_acc, all_prec, all_rec, all_f1, all_f1_w = [], [], [], [], []
for i in range(len(history)):
    all_acc.append(history[i].get('accuracy'))
    all_prec.append(history[i].get('precision_macro'))
    all_rec.append(history[i].get('recall_macro'))
    all_f1.append(history[i].get('f1_macro'))
    all_f1_w.append(history[i].get('f1_weighted'))

plt.figure(figsize=(15,12))
plt.xlabel("Nodi")
plt.ylabel("Value %")
plt.xticks(xs)
#plt.plot(xs, accuracy_local_list)
plt.plot(
    all_acc,
    label="Gossip Accuracy",
    color="royalblue",
    linewidth=2,
)
plt.plot(
    all_prec,
    label="Gossip Precision",
    color="limegreen",
    linewidth=2,
)
plt.plot(
    all_rec,
    label="Gossip Recall",
    color="darkviolet",
    linewidth=2,
)
plt.plot(
    all_f1,
    label="Gossip F1",
    color="crimson",
    linewidth=2,
)
plt.plot(
    all_f1_w,
    label="Gossip F1 Weighted",
    color="black",
    linewidth=2,
)

plt.legend()
plt.savefig(Path(path_folder) / "comparison_plot.jpg")
plt.show()
plt.close()

In [ ]:
import statistics
all_value_means = []
#alphas = [0.05, 0.1, 0.3, 0.5, 0.7]
alphas = [3,5,7,9]

all_value_means.append(statistics.mean(all_acc))
all_value_means.append(statistics.mean(all_prec))
all_value_means.append(statistics.mean(all_rec))
all_value_means.append(statistics.mean(all_f1))
all_value_means.append(statistics.mean(all_f1_w))

print(all_value_means)

In [ ]:
metrics_history = {
        "N_nodes": num_nodes,
        "Connectivity" : k,
        "Strategy": aggr,
        "accuracy": all_value_means[0],
        "precision_macro": all_value_means[1],
        "recall_macro": all_value_means[2],
        "f1_macro": all_value_means[3],
        "f1_weighted": all_value_means[4],
    }

file_path = path_folder / "report_avg.json"
if not file_path.exists():
    with open(file_path, "w") as outfile:
         json.dump(metrics_history, outfile, indent=1)
    outfile.close